# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 제출 정보
- 이름: 전해린 
- GitHub ID: Haerin03
- 작성일: 2026.10.07
- 최종 제출 URL: https://github.com/Haerin03/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [72]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'chapter05':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\chapter05
프로젝트 루트: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study
원본 데이터 폴더: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\data\raw
전처리 데이터 폴더: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\data\processed
보고서 폴더: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [73]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [74]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [75]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [76]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [78]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [79]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 1) 전처리 전 상태 기록
- 각 데이터 shape: 
```text
customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)
```
- 결측치: 
```text
[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64
```
- 전체 중복: 
```text
customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0
```
- 주요 ID 중복: 
```text
customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0
```
- 타입 문제 후보: 
```text
문자열: 앞뒤 공백 제거, 같은 의미의 문자열 같은 표기로 통일
날짜 칼럼: 날짜형으로 변환
숫자 칼럼: 가격, 수량, 단가 등의 칼럼 숫자형으로 변환
```

![전처리 전 상태](images/step01_before.png)

### 결과 관찰

customers 150행, products 100행, orders 300행, order_items 764행으로 구성되어 있었다. 각 데이터셋의 결측치를 확인한 결과 모든 칼럼의 결측치가 0건이었으며, 전체 행 기준 중복과 customer_id, product_id, order_id, order_item_id 등 주요 ID 기준 중복도 발견되지 않았다.

따라서 현재 데이터에서는 결측치와 중복으로 인한 직접적인 데이터 손실 문제는 확인되지 않았다. 다만 문자열의 앞뒤 공백이나 동일한 의미를 가진 범주값의 표기 차이, 날짜 및 숫자 칼럼의 데이터 타입과 같이 단순 결측치 및 중복 검사만으로 확인하기 어려운 문제는 추가적인 전처리가 필요하다고 판단하였다.

### 나의 해석과 판단
어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요.

결측치나 중복을 바로 제거하기보다는 먼저 문자열과 데이터 타입을 정리하고, 변환 과정에서 새롭게 발생하는 결측이나 변환 실패가 있는지를 확인하는 것이 중요하다고 판단하였다.

특히 주문 상태처럼 동일한 의미가 여러 문자열로 표현되어 있으면 이후 범주별 집계 결과가 분산될 수 있으므로 표기를 통일할 필요가 있다. 또한 가격, 수량, 단가와 같은 수치형 데이터와 주문일자 및 가입일자와 같은 날짜 데이터는 분석에 적절한 타입으로 변환한 뒤 변환 실패 여부를 확인하는 순서로 처리하고자 하였다.

### 업무·분석적 의미

전처리 전 데이터 상태를 기록하면 이후 분석 결과가 원본 데이터의 특성 때문인지 전처리 과정에서 발생한 변화 때문인지 구분할 수 있다. 또한 전처리 전후의 행 수, 결측치, 중복 등을 비교할 수 있어 데이터가 의도하지 않게 삭제되거나 변경되지 않았는지 확인하는 기준이 된다.

### 한계와 추가 확인 사항

결측치와 중복이 0건이라는 사실만으로 데이터 품질이 완전히 확보되었다고 판단할 수는 없다. 문자열 내부의 표기 차이, 잘못된 날짜나 숫자 값, 현실적으로 가능한 범위를 벗어난 값 등은 별도로 확인해야 한다. 또한 실제 업무 데이터라면 원본 시스템의 입력 규칙과 수집 과정도 함께 확인할 필요가 있다.

## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [80]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [81]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [82]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [83]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [84]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\82105\AppData\Local\Temp\ipykernel_3444\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\82105\AppData\Local\Temp\ipykernel_3444\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user

In [85]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [86]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [87]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


,order_date,order_month,order_dayofweek
0,2026-05-07,2026-05,Thursday
1,2025-07-23,2025-07,Wednesday
2,2025-11-19,2025-11,Wednesday
3,2026-01-30,2026-01,Friday
4,2025-12-21,2025-12,Sunday


In [88]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-06-19
1,2,2025-11-02
2,3,2024-06-12
3,4,2026-04-13
4,5,2024-09-13


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [89]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 2) 문자열·타입·결측 처리
- 적용한 문자열 정리: 문자열 앞뒤 공백 제거, 주문 상태값 통일
- 숫자 변환 대상과 실패 건수: price, quantity, unit_price
```text
price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0
```
- 날짜 변환 대상과 실패 건수: order_date, signup_date 
```text
order_date 변환 실패: 0
signup_date 변환 실패: 0
```
- 결측 처리 기준: 나이와 같은 숫자형 칼럼은 중앙값으로 대체, 도시와 같은 범주형 칼럼은 Unknown으로 대체

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

문자열 칼럼의 앞뒤 공백을 제거하고 주문 상태값의 표현을 통일하였다. price, quantity, unit_price를 숫자형으로 변환한 결과 변환 실패는 발생하지 않았으며, order_date와 signup_date의 날짜형 변환에서도 실패가 발생하지 않았다.

결측치 처리 기준으로 숫자형인 age는 중앙값으로 대체하고, 범주형인 city는 결측 여부를 명확하게 구분할 수 있도록 'Unknown'으로 대체하도록 구성하였다.

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.

age의 경우 평균보다 이상값의 영향을 적게 받는 중앙값을 사용하여 결측치를 대체하도록 하였다. city와 같은 범주형 변수는 임의의 특정 도시로 대체할 경우 잘못된 정보를 만들 수 있으므로 'Unknown'이라는 별도 범주를 사용하는 것이 적절하다고 판단하였다.

다만 결측값 대체는 실제 관측값을 복원하는 것이 아니라 분석을 위한 가정을 추가하는 과정이다. 따라서 결측치가 많을 경우 중앙값이나 Unknown으로 일괄 대체하면 원래 데이터의 분포와 범주 비율이 달라질 수 있다. 실제 업무에서는 결측 발생 원인을 먼저 확인한 뒤 삭제 또는 대체 여부를 결정할 필요가 있다.

### 업무·분석적 의미

데이터 타입과 문자열 표현을 통일하면 이후 집계와 계산 과정에서 발생할 수 있는 오류를 줄일 수 있다. 특히 숫자형 변환은 매출 및 주문 금액 계산에 필요하고, 날짜형 변환은 월별 주문량이나 요일별 주문 패턴과 같은 시계열 분석을 가능하게 한다. 범주값 표준화는 같은 의미의 데이터가 서로 다른 범주로 집계되는 문제를 방지한다.

### 한계와 추가 확인 사항

현재 데이터에서는 숫자 및 날짜 변환 실패가 발견되지 않았기 때문에 변환 로직이 다양한 오류 데이터를 모두 처리할 수 있다고 단정할 수 없다. 또한 중앙값 및 Unknown 대체 방식은 실습을 위한 기준이며, 실제 업무에서는 결측 발생 원인과 데이터 수집 규칙을 확인한 뒤 적절한 처리 방법을 결정해야 한다.

## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [90]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [91]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [92]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 3) 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: customer_id, product_id, order_id, order_item_id 등 주요 키의 중복을 기준으로 판단한 완전 중복 행 제거
- 표준화한 범주값: 주문 상태값이 산발적으로 표현되어 있다면 서로 같은 상태로 집계할 수 없기 때문에 대표 표기로 통일함
```text
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}
```
- 허용값 밖 값: 
```text
price가 음수인 상품
주문 상태값이 completed, cancelled, refunded 이외의 다른 값인 주문 
unit_price가 음수인 주문 상세 
```
- 이상값 후보:
```text
price가 0 이하인 상품
quantity가 0 이하인 주문 상세
unit_price가 0 이하인 주문 상세 
```

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

전체 행 중복과 주요 ID 기준 중복을 확인한 결과 모든 데이터셋에서 중복이 0건으로 확인되었다. 주문 상태값은 completed, cancelled, refunded를 기준으로 같은 의미의 다양한 표현을 하나의 값으로 통일하도록 하였다.

또한 price, quantity, unit_price가 0 이하인 경우를 정상적인 분석 대상에서 제외하도록 전처리 기준을 설정하였다. 이를 통해 이후 금액 및 주문량 계산에 부적절한 값이 포함되는 것을 방지하였다.

### 나의 해석과 판단
이상값을 자동 삭제하지 않고 후보로 검토해야 하는 이유를 작성하세요.

이상값은 단순히 값이 크거나 작다는 이유만으로 자동 삭제해서는 안 된다고 판단하였다. 실제 업무에서는 0원 상품, 무료 증정품, 반품이나 정정 거래 등 일반적인 범위를 벗어나더라도 업무적으로 의미가 있는 값이 존재할 수 있기 때문이다.

따라서 이상값은 먼저 후보로 식별한 뒤 해당 값이 입력 오류인지 실제 업무에서 발생 가능한 값인지 확인하고 처리 여부를 결정하는 것이 안전하다. 이번 실습에서는 price, quantity, unit_price가 0 이하인 경우를 정상 분석 대상에서 제외하는 명시적인 기준을 적용하였다.

### 업무·분석적 의미

범주값을 표준화하면 동일한 주문 상태가 여러 범주로 분리되어 집계되는 문제를 방지할 수 있다. 또한 가격, 수량, 단가의 비정상적인 값을 관리하면 매출 합계나 평균 주문 금액과 같은 지표가 잘못 계산될 가능성을 줄일 수 있다.

### 한계와 추가 확인 사항

현재 적용한 0 이하 값 제외 기준이 모든 실제 업무 상황에서 적절하다고 볼 수는 없다. 실제 데이터에서는 무료 상품이나 반품 거래처럼 0 또는 음수 값이 의미를 가질 가능성이 있으므로 업무 규칙을 추가로 확인해야 한다. 주문 상태 역시 현재 정의한 세 가지 상태 외에 정상적인 다른 상태가 존재할 가능성을 확인할 필요가 있다.

## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [93]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [94]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [95]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 4) PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 0
- 없는 `order_id`: 0
- 없는 `product_id`: 0
```text
index  check	invalid_count
0	orders.customer_id exists in customers.customer_id	0
1	order_items.order_id exists in orders.order_id	0
2	order_items.product_id exists in products.product_id	0
```
- 생성한 파생 컬럼:
```text
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)
```
- `line_total` 검증: 전처리 후 주문 상세 금액 합계: 255610000

변환 전 quantity, unit_price의 숫자형 변환 후 변환 실패를 확인 후 계산
```text
order_items_clean["quantity"] = pd.to_numeric(
    order_items_clean["quantity"],
    errors="coerce")

order_items_clean["unit_price"] = pd.to_numeric(
    order_items_clean["unit_price"],
    errors="coerce")

print("quantity 결측/변환 실패:", order_items_clean["quantity"].isna().sum())

print("unit_price 결측/변환 실패:", order_items_clean["unit_price"].isna().sum())
```

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

전처리 후 PK/FK 관계를 다시 확인한 결과 orders의 customer_id, order_items의 order_id, order_items의 product_id에서 참조 대상이 존재하지 않는 경우는 모두 0건이었다. 따라서 전처리 이후에도 데이터셋 간 참조 관계가 유지되고 있음을 확인하였다.

또한 quantity와 unit_price를 숫자형으로 변환한 후 line_total을 계산하였으며, 전처리된 주문 상세 데이터의 line_total 합계는 255,610,000으로 확인되었다.

### 나의 해석과 판단
전처리 후 관계를 다시 검증해야 하는 이유를 작성하세요.

전처리 과정에서 행을 제거하거나 ID 값이 변경되면 기존에 정상적이었던 데이터셋 간 관계가 깨질 수 있기 때문에 전처리 후 PK/FK 관계를 다시 검증해야 한다.
예를 들어 customers에서 특정 고객을 제거했지만 해당 고객의 주문이 orders에 남아 있다면 고객 정보와 주문 정보를 정상적으로 연결할 수 없다. 따라서 전처리가 완료된 데이터만을 대상으로 관계 검증을 다시 수행하는 것이 필요하다고 판단하였다.

또한 line_total은 기존 데이터가 아니라 quantity와 unit_price를 이용해 생성한 값이므로 두 원본 칼럼의 숫자형 변환 성공 여부를 확인한 후 계산해야 신뢰할 수 있다.

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [96]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 5) 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 150, 100, 300, 764 | 150, 100, 300, 764 | 제거 대상 행이 없어 기존 행 수가 유지됨 |
| 결측 | 0 | 0 | 확인된 결측치가 없어 건수 변화 없음 |
| 중복 | 0 | 0 | 전체 행 및 주요 ID 기준 중복이 발견되지 않음 |
| 변환 실패 | 0 | 0 | 숫자 및 날짜 변환 과정에서 실패가 발생하지 않음 |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

전처리 전후를 비교한 결과 네 데이터셋 모두 행 수가 동일하게 유지되었다. 결측치와 중복 역시 전처리 전후 모두 0건이었으며 숫자 및 날짜 변환 과정에서도 실패가 발생하지 않았다.

다만 orders에는 order_month와 order_dayofweek가 추가되어 칼럼 수가 5개에서 7개로 증가하였고, order_items에는 line_total이 추가되어 5개에서 6개로 증가하였다. 이는 데이터 손실이 아니라 분석을 위한 파생 컬럼 생성에 따른 변화이다.

### 나의 해석과 판단
행 수나 값 분포 변화가 의도한 결과인지 판단한 근거를 작성하세요.

전처리 전후 행 수가 동일하다는 점에서 이번 데이터에서는 전처리 규칙에 의해 제거된 행이 없음을 확인할 수 있다. 또한 결측, 중복 및 변환 실패가 발생하지 않았으므로 원본 관측치를 불필요하게 삭제하지 않고 필요한 타입 및 표현 정리와 파생변수 생성을 수행한 것으로 판단하였다.

칼럼 수의 증가는 order_month, order_dayofweek, line_total과 같은 분석 목적의 파생변수를 생성한 결과이므로 의도한 변화라고 판단하였다.

### 업무·분석적 의미

전처리 전후의 데이터 크기와 품질 지표를 비교하면 전처리 과정에서 데이터가 예상과 다르게 손실되었는지 확인할 수 있다. 이번 결과에서는 행 수를 유지하면서 분석에 필요한 날짜 및 금액 관련 파생변수를 추가하였기 때문에 이후 월별 주문 분석, 요일별 주문 분석, 주문 금액 분석에 활용할 수 있는 형태로 데이터가 정리되었다.

### 한계와 추가 확인 사항

행 수와 결측치 수가 동일하다는 것만으로 모든 값이 올바르게 처리되었다고 판단할 수는 없다. 전처리 과정에서 범주값이 의도한 형태로 표준화되었는지, 파생변수가 정확하게 계산되었는지, 값의 분포가 비정상적으로 변하지 않았는지 등을 EDA 단계에서 추가로 확인할 필요가 있다.


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [97]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/Users/82105/Desktop/대5/llm-data-analysis-study/llm-data-analysis-study/data/processed/customers_clean.csv'),
 WindowsPath('c:/Users/82105/Desktop/대5/llm-data-analysis-study/llm-data-analysis-study/data/processed/orders_clean.csv'),
 WindowsPath('c:/Users/82105/Desktop/대5/llm-data-analysis-study/llm-data-analysis-study/data/processed/order_items_clean.csv'),
 WindowsPath('c:/Users/82105/Desktop/대5/llm-data-analysis-study/llm-data-analysis-study/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [98]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\Users\82105\Desktop\대5\llm-data-analysis-study\llm-data-analysis-study\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [99]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [100]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


C:\Users\82105\AppData\Local\Temp\ipykernel_3444\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\82105\AppData\Local\Temp\ipykernel_3444\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [101]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [102]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [103]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 6) 재현 가능한 전처리 확인
- 생성된 clean CSV:
```text
  - `data/processed/customers_clean.csv`
  - `data/processed/products_clean.csv`
  - `data/processed/orders_clean.csv`
  - `data/processed/order_items_clean.csv`
```
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md
- `python scripts/preprocess_data.py` 재실행 결과:
```text
  - 오류 없이 `전처리 완료` 메시지가 출력되었다.
  - 전처리된 4개의 CSV 파일과 요약 보고서가 정상적으로 생성되었다.
  - 전처리 전후 행 수, 중복 여부, PK/FK 관계 검증 결과가 함께 출력되었다.
```
- 재실행 후 달라진 점: 동일한 Raw 데이터를 사용하여 다시 실행했을 때 행 수, 중복 점검, 관계 점검 결과가 동일하게 재현되었다. 기존 Raw 데이터는 수정하지 않고 data/processed에 전처리 결과를 다시 생성하므로 원본 데이터는 유지되었다.

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
Notebook에서 한 번 성공하는 것과 스크립트로 재현 가능한 것의 차이를 작성하세요.

Notebook에서 한 번 전처리에 성공하는 것만으로는 동일한 결과가 다시 만들어진다는 것을 보장하기 어렵다. Notebook은 셀의 실행 순서나 이전 실행 결과에 따라 상태가 달라질 수 있기 때문이다.

반면 전처리 과정을 preprocess_data.py와 같은 스크립트로 구성하면 동일한 Raw 데이터에 동일한 전처리 규칙을 순서대로 적용할 수 있다. 따라서 다른 환경이나 이후 시점에서도 같은 명령으로 전처리 결과를 다시 생성하고 검증할 수 있다는 장점이 있다.

이번 실습에서는 전처리 결과 파일뿐 아니라 전처리 전후 크기, 중복 점검, 데이터 간 관계 점검 결과와 요약 보고서까지 함께 생성하도록 하여 전처리 과정의 재현성과 확인 가능성을 높였다.

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


### 검토 결과

(1) 원본 데이터 직접 수정하고 있음

```text
customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()```

customers의 age를 직접 변경하고 있음 -> 원본을 copy한 후에 수정하는 것이 안전함

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

(2) 결측치 처리 방식 - 평균 대체의 부족함
평균은 극단값 영향을 받고, 소숫점까지 계산된다면 나이도 소수로 채워지는 문제가 발생함
단순 전처리 과정에서는 중앙값이 더욱 적합하다고 판단
(결측 원인 파악 후 결측치를 평균, 중앙값 등 어떤 값으로 대체할지 판단하는 것이 가장 적합함)

age_median = customers_clean["age"].median()
customers_clean["age"] = customers_clean["age"].fillna(age_median)

(3) 날짜 변환 실패 확인
날짜 변환 실패를 확인하지 않음
안전하게 아래와 같이 변환

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce")

아래와 같이 실패 건수 확인

invalid_dates = orders_clean["order_date"].isna().sum()

print("날짜 변환 실패 건수:", invalid_dates)

(4) 중복 제거 기준 불명확

customers = customers.drop_duplicates()

모든 열이 완전히 동일한 행만 제거함
만약 customer_id가 동일하지만, 다른 속성값이 다르게 표시되면 제거되지 않음
customer_id와 같은 고유 식별자를 기준으로 먼저 중복을 확인하는 것이 적합함

duplicate_customer_ids = customers_clean.duplicated(
    subset=["customer_id"],
    keep=False)

print(
    "중복 customer_id 행 수:",
    duplicate_customer_ids.sum())

중복이 실제 오류임을 확인한 뒤에만 아래와 같이 처리해야 함

customers_clean = customers_clean.drop_duplicates(
    subset=["customer_id"],
    keep="first")

즉, 무조건 drop_duplicates()를 하는 것보다 중복을 확인한 후에 처리 기준을 결정하는 것이 안전함

(5) line_total 계산 전 숫자형 변환이 필요함

order_items_clean["quantity"] = pd.to_numeric(
    order_items_clean["quantity"],
    errors="coerce")

order_items_clean["unit_price"] = pd.to_numeric(
    order_items_clean["unit_price"],
    errors="coerce")

그 후에 변환 실패를 확인함

print(
    "quantity 변환 실패/결측:",
    order_items_clean["quantity"].isna().sum())

print(
    "unit_price 변환 실패/결측:",
    order_items_clean["unit_price"].isna().sum())

마지막으로 계산해야 안전함

order_items_clean["line_total"] = (
    order_items_clean["quantity"]
    * order_items_clean["unit_price"])

(6) 안전한 코드로 수정

단순히 코드가 에러 없이 실행되는 것이 아니라 전처리 과정에서 데이터가 잘못 바뀌었는지를 변환 실패 확인, 중복 확인 등을 실행하면서 수정함

In [105]:
# 1. 원본 데이터 보존

customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

# 2. customers 전처리

# age를 숫자형으로 변환
customers_clean["age"] = pd.to_numeric(
    customers_clean["age"],
    errors="coerce")

# 결측치는 중앙값으로 대체
age_median = customers_clean["age"].median()

customers_clean["age"] = (
    customers_clean["age"]
    .fillna(age_median))

# 4. customer_id 중복 확인

duplicate_customer_ids = customers_clean.duplicated(
    subset=["customer_id"],
    keep=False)

print("중복 customer_id 행 수:", duplicate_customer_ids.sum())

# 중복 제거가 적절하다고 판단한 경우
customers_clean = customers_clean.drop_duplicates(
    subset=["customer_id"],
    keep="first")

# 3. orders 날짜 변환

# 날짜형 변환
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce")

# 날짜형 변환 실패 확인
invalid_dates = orders_clean["order_date"].isna().sum()

print("날짜 변환 실패 건수:", invalid_dates)

# 5. order_items 전처리

order_items_clean["quantity"] = pd.to_numeric(
    order_items_clean["quantity"],
    errors="coerce")

order_items_clean["unit_price"] = pd.to_numeric(
    order_items_clean["unit_price"],
    errors="coerce")

print("quantity 결측/변환 실패:", order_items_clean["quantity"].isna().sum())

print("unit_price 결측/변환 실패:", order_items_clean["unit_price"].isna().sum())

order_items_clean["line_total"] = (
    order_items_clean["quantity"]
    * order_items_clean["unit_price"])

중복 customer_id 행 수: 0
날짜 변환 실패 건수: 0
quantity 결측/변환 실패: 0
unit_price 결측/변환 실패: 0


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [68]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_tables = []

for name, df in clean_data.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={'index': 'column'})
    summary.insert(0, 'dataset', name)

    missing_tables.append(summary)

missing_table = pd.concat(missing_tables, ignore_index=True)

missing_table

,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [69]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

customers_clean['age'] = to_number(customers_clean['age'])
print('age 변환 실패:', customers_clean['age'].isna().sum())

print('18세 미만인 고객 수:', customers_clean[customers_clean['age'] < 18].shape[0])
print('100세 초과인 고객 수:', customers_clean[customers_clean['age'] > 100].shape[0])

age 변환 실패: 0
18세 미만인 고객 수: 0
100세 초과인 고객 수: 0


In [70]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

monthly_order_count = (
    orders_clean
    .groupby('order_month')
    .size()
    .reset_index(name='order_count')
    .sort_values('order_month')
)

monthly_order_count

,order_month,order_count
0,2025-09,8
1,2025-10,23
2,2025-11,22
3,2025-12,31
4,2026-01,34
5,2026-02,15
6,2026-03,38
7,2026-04,25
8,2026-05,26
9,2026-06,26


In [71]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.

top_10_line_total = (
    order_items_clean
    .sort_values('line_total', ascending=False)
    .head(10)
)

top_10_line_total

,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
493,494,190,70,5,198000,990000
78,79,35,70,5,198000,990000
631,632,247,58,5,197000,985000
604,605,235,58,5,197000,985000
113,114,51,58,5,197000,985000
569,570,221,58,5,197000,985000
45,46,21,58,5,197000,985000


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 7) Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. Raw 데이터는 직접 수정하지 않고 복사본을 만들어 전처리하며, 처리 전후 결과를 비교한다.
2. 결측치, 중복, 이상값을 발견했다고 바로 삭제하지 않고 데이터의 의미와 업무 규칙을 확인한 뒤 처리 기준을 정한다.
3. 문자열, 숫자, 날짜 등의 타입을 정리한 뒤 변환 실패 여부와 PK/FK 관계를 다시 검증하여 전처리 과정에서 새로운 문제가 발생하지 않았는지 확인한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

가장 위험하다고 생각한 자동 처리는 이상값을 조건만으로 자동 삭제하는 것이다. 예를 들어 가격이나 수량이 0 이하라는 이유만으로 행을 삭제하면 실제 업무에서는 무료 상품, 반품, 취소 또는 데이터 정정과 같이 의미가 있는 거래를 제거할 가능성이 있다. 따라서 이상값은 우선 후보로 확인하고 실제 업무 규칙과 원본 데이터의 의미를 확인한 뒤 삭제 여부를 결정해야 한다.

### 다음 EDA에서 특히 주의할 데이터 특성

다음 EDA에서는 주문 상태별 데이터 분포와 월별·요일별 주문량의 차이를 확인하고, 특정 상품이나 고객에 주문이 과도하게 집중되어 있는지 살펴볼 필요가 있다. 또한 quantity, unit_price, line_total과 같은 수치형 변수에서 극단값이 존재하는지 확인하여 평균이나 총액이 일부 큰 값에 의해 과도하게 영향을 받는지도 검토할 필요가 있다.

### 현재 전처리의 한계

현재 전처리는 실습 데이터에 맞추어 정의한 규칙이므로 실제 업무 데이터에 그대로 적용하기에는 한계가 있다. 결측치를 중앙값이나 Unknown으로 대체하는 기준과 0 이하의 가격·수량·단가를 제외하는 기준은 실제 업무 규칙을 충분히 반영하지 못할 수 있다.

또한 현재 데이터에서 변환 실패나 PK/FK 문제가 발견되지 않았기 때문에 다양한 오류 상황에서 전처리 코드가 적절하게 동작하는지는 추가적인 검증이 필요하다. 실제 업무에서는 원본 시스템의 입력 규칙, 데이터 수집 과정, 담당자의 업무 지식을 함께 확인하여 전처리 기준을 보완해야 한다.

## 최종 제출 체크
- [O] raw 원본을 덮어쓰지 않았습니다.
- [O] 처리 전/후 비교를 남겼습니다.
- [O] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [O] clean CSV와 재실행 결과를 확인했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.